# Linear Regression with Statsmodels

This notebook follows one continuous-target regression example from data generation through two kinds of evaluation:

1. **Training fit:** fit and evaluate on all generated rows so the OLS statistics and manual R² calculation are visible.
2. **Held-out evaluation:** refit on a training partition and evaluate on unseen test rows to measure generalization.

The example uses six features, but the idea is the same as `y_hat = b0 + b1*x`: learn coefficients that minimize squared residuals.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Fixed seed makes the generated dataset reproducible.
np.random.seed(42)

n_samples = 500
n_features = 6
feature_names = [f"feature_{i}" for i in range(1, n_features + 1)]

X_train_raw = pd.DataFrame(
    np.random.randn(n_samples, n_features),
    columns=feature_names,
)

true_intercept = 3.0
true_weights = np.array([1.8, -1.2, 0.7, 2.4, -0.5, 1.1])
noise = np.random.normal(loc=0.0, scale=1.0, size=n_samples)

y_train = pd.Series(
    true_intercept + X_train_raw.values @ true_weights + noise,
    name="label",
)

print("X_train_raw shape:", X_train_raw.shape)
print("y_train shape    :", y_train.shape)

X_train_raw shape: (500, 6)
y_train shape    : (500,)


## Fit an OLS model

Ordinary Least Squares chooses coefficients that minimize `sum((y - y_hat) ** 2)`.

Statsmodels' matrix API does not add an intercept automatically. `sm.add_constant()` adds a column of ones so the model can learn `b0` instead of being forced through the origin. `sm.OLS(...).fit()` then solves the least-squares problem using stable linear-algebra routines.

In [2]:
X_train = sm.add_constant(X_train_raw, has_constant="add")

ols_model = sm.OLS(endog=y_train, exog=X_train)
ols_results = ols_model.fit()

print(ols_results.summary())

                            OLS Regression Results                            
Dep. Variable:                  label   R-squared:                       0.921
Model:                            OLS   Adj. R-squared:                  0.920
Method:                 Least Squares   F-statistic:                     962.8
Date:                Fri, 27 Nov 2026   Prob (F-statistic):          1.53e-268
Time:                        10:29:52   Log-Likelihood:                -718.63
No. Observations:                 500   AIC:                             1451.
Df Residuals:                     493   BIC:                             1481.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9972      0.046     64.645      0.0

## Evaluate training fit

R² compares the fitted model with the constant model that predicts `mean(y)`:

```text
R² = 1 - SS_res / SS_tot
SS_res = sum((y - y_hat)²)
SS_tot = sum((y - mean(y))²)
```

For **in-sample OLS with an intercept**, R² is between 0 and 1 because the fitted model cannot do worse on its training rows than the intercept-only solution. Adjusted R² also accounts for the number of predictors. RMSE and MAE express error in the target's units.

This range is not a universal rule: held-out R² can be negative, as explained below. R² is a regression metric, not a generic classification metric.

In [3]:
y_train_pred = ols_results.predict(X_train)

r2_native = ols_results.rsquared
adj_r2_native = ols_results.rsquared_adj

# Manual calculation verifies the native statsmodels R-squared value.
ss_res = np.sum((y_train - y_train_pred) ** 2)
ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
r2_manual = 1.0 - (ss_res / ss_tot)

n = len(y_train)
p = X_train_raw.shape[1]
adj_r2_manual = 1.0 - (1.0 - r2_manual) * ((n - 1) / (n - p - 1))

rmse = np.sqrt(np.mean((y_train - y_train_pred) ** 2))
mae = np.mean(np.abs(y_train - y_train_pred))

print(f"R-squared (ols_results.rsquared)    : {r2_native:.4f}")
print(f"R-squared (manual verification)     : {r2_manual:.4f}")
print(f"Adjusted R-squared (rsquared_adj)   : {adj_r2_native:.4f}")
print(f"Adjusted R-squared (manual)         : {adj_r2_manual:.4f}")
print(f"Training RMSE                       : {rmse:.4f}")
print(f"Training MAE                        : {mae:.4f}")

R-squared (ols_results.rsquared)    : 0.9214
R-squared (manual verification)     : 0.9214
Adjusted R-squared (rsquared_adj)   : 0.9204
Adjusted R-squared (manual)         : 0.9204
Training RMSE                       : 1.0185
Training MAE                        : 0.8256


## Evaluate generalization on held-out rows

Training performance measures fit to known rows; test performance measures how the learned relationship generalizes. The next cells split the same synthetic dataset, fit only on 80% of the rows, and predict the remaining 20%.

The baseline predicts `y_fit.mean()` for every test row. It uses training information only, so its held-out R² need not be exactly zero. A held-out R² below zero means predictions are worse than the constant comparator used by the R² definition; unlike in-sample OLS R² with an intercept, negative values are possible.

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X_fit_raw, X_test_raw, y_fit, y_test = train_test_split(
    X_train_raw,
    y_train,
    test_size=0.2,
    random_state=42,
)

X_fit = sm.add_constant(X_fit_raw, has_constant="add")
X_test = sm.add_constant(X_test_raw, has_constant="add")

heldout_results = sm.OLS(endog=y_fit, exog=X_fit).fit()
y_test_pred = heldout_results.predict(X_test)

print("Training partition shape:", X_fit_raw.shape)
print("Test partition shape    :", X_test_raw.shape)

Training partition shape: (400, 6)
Test partition shape    : (100, 6)


In [5]:
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

# A deployable constant baseline can use the training target mean, not test labels.
baseline_pred = np.full(y_test.shape, y_fit.mean())
baseline_r2 = r2_score(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Test R-squared                      : {test_r2:.4f}")
print(f"Test RMSE                           : {test_rmse:.4f}")
print(f"Test MAE                            : {test_mae:.4f}")
print("\nTraining-mean baseline on the same test rows")
print(f"Baseline R-squared                  : {baseline_r2:.4f}")
print(f"Baseline RMSE                       : {baseline_rmse:.4f}")
print(f"Baseline MAE                        : {baseline_mae:.4f}")

Test R-squared                      : 0.8882
Test RMSE                           : 1.0876
Test MAE                            : 0.8882

Training-mean baseline on the same test rows
Baseline R-squared                  : -0.0057
Baseline RMSE                       : 3.2624
Baseline MAE                        : 2.6112
